# Train masked vertebral morphology with XGBoost

This notebook is the inspectable, step-by-step version of `src/train_masked_morphology_xgboost.py`. It trains five residual regressors from four context vertebrae: `m2`, `m1`, `p1`, and `p2`.

> Research use only. The current task reconstructs observed masked morphology from the training population; it is not a clinically validated normality or diagnosis model.

## Training flow

```text
Load fixed train / validation / test tables
                    ↓
Validate 28 context features and group separation
                    ↓
Measure nearest-neighbor interpolation baseline
                    ↓
Train one residual XGBoost model per output
                    ↓
Evaluate validation and locked test partitions
                    ↓
Inspect features, sources, and largest errors
                    ↓
Save models and reports
```

## 1. Environment

Uncomment the next line only if the project environment has not been installed.

In [ ]:
# %pip install -r ../../requirement.txt

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from xgboost import XGBRegressor

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "dataset").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project repository.")

REPO_ROOT = find_repo_root()

DATASET_ROOT = REPO_ROOT / "dataset/processed/masked_morphology_coco_nih_lumos"
OUTPUT_DIR = REPO_ROOT / "outputs/masked_morphology/xgboost_four_context_notebook_v1"
SEED = 20260813
LIMIT_PER_SPLIT = None  # Set to 200 for a quick smoke run.
SAVE_ARTIFACTS = True

print("Repository:", REPO_ROOT)
print("Dataset:", DATASET_ROOT)
print("Artifacts:", OUTPUT_DIR)

### Inspectable data and evaluation helpers

These cells replace the helper imports from the command-line script. Each model predicts a **residual**: the correction to the simple interpolation baseline. The reconstructed value is baseline + predicted residual.

In [ ]:
SPLITS = ("train", "val", "test")

@dataclass(frozen=True)
class OutputContract:
    name: str
    residual_column: str
    actual_column: str
    baseline_column: str
    scale_column: str | None
    unit: str

OUTPUT_CONTRACTS = {
    name: OutputContract(
        name=name,
        residual_column=f"y_{name}_residual",
        actual_column=f"y_{name}_norm",
        baseline_column=f"baseline_{name}_norm",
        scale_column="reference_height_px" if "height" in name else "reference_width_px",
        unit="normalized",
    )
    for name in ("left_height", "right_height", "superior_width", "inferior_width")
}
OUTPUT_CONTRACTS["orientation"] = OutputContract(
    name="orientation",
    residual_column="y_orientation_residual_deg",
    actual_column="y_orientation_deg",
    baseline_column="baseline_orientation_deg",
    scale_column=None,
    unit="degrees",
)

display(pd.DataFrame([vars(contract) for contract in OUTPUT_CONTRACTS.values()]))

In [ ]:
def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as file:
        payload = json.load(file)
    if not isinstance(payload, dict):
        raise ValueError(f"Expected a JSON object in {path}")
    return payload


def load_dataset(dataset_root: Path, *, limit_per_split: int | None = None):
    root = dataset_root.resolve(strict=True)
    schema = load_json(root / "feature_schema.json")
    features = [str(value) for value in schema.get("numeric_input_columns", [])]
    if not features:
        raise ValueError("feature_schema.json contains no numeric input columns")
    if any(name.startswith(("y_", "baseline_")) for name in features):
        raise ValueError("Target or baseline leakage exists in numeric_input_columns")

    required = set(features)
    required.update({
        "sample_id", "group_id", "sample_weight", "source_dataset",
        "source_dataset_key", "view", "image_path", "target_annotation_id",
        "target_chain_rank", "chain_count", "reference_height_px",
        "reference_width_px",
    })
    for contract in OUTPUT_CONTRACTS.values():
        required.update({
            contract.residual_column,
            contract.actual_column,
            contract.baseline_column,
        })

    frames = {}
    for split in SPLITS:
        path = root / split / "masked_samples.csv"
        frame = pd.read_csv(path, low_memory=False)
        if limit_per_split is not None:
            frame = frame.head(limit_per_split).copy()
        missing = sorted(required - set(frame.columns))
        if missing:
            raise ValueError(f"{path} is missing columns: {missing}")
        if frame.empty:
            raise ValueError(f"{path} contains no rows")

        numeric = features + [c.residual_column for c in OUTPUT_CONTRACTS.values()]
        numeric += ["sample_weight", "reference_height_px", "reference_width_px"]
        if not np.isfinite(frame[numeric].to_numpy(dtype=np.float64)).all():
            raise ValueError(f"{path} contains non-finite model values")
        if (frame["sample_weight"] <= 0.0).any():
            raise ValueError(f"{path} contains nonpositive sample weights")
        if frame["sample_id"].duplicated().any():
            raise ValueError(f"{path} contains duplicate sample IDs")
        frames[split] = frame

    for first, second in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = set(frames[first]["group_id"]) & set(frames[second]["group_id"])
        if overlap:
            raise ValueError(
                f"Group leakage between {first} and {second}: {len(overlap)} groups"
            )
    return frames, schema, features

In [ ]:
def wrap_axial_deg(angle):
    """Wrap an undirected vertebral angle to [-90, 90) degrees."""
    return (np.asarray(angle, dtype=np.float64) + 90.0) % 180.0 - 90.0


def absolute_residual_error(contract, actual_residual, predicted_residual):
    difference = np.asarray(actual_residual) - np.asarray(predicted_residual)
    if contract.name == "orientation":
        return np.abs(wrap_axial_deg(difference))
    return np.abs(difference)


def weighted_r2(actual, predicted, weights):
    actual = np.asarray(actual, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    actual_mean = float(np.average(actual, weights=weights))
    residual_sum = float(np.sum(weights * np.square(actual - predicted)))
    total_sum = float(np.sum(weights * np.square(actual - actual_mean)))
    if total_sum <= np.finfo(np.float64).eps:
        return 1.0 if residual_sum <= np.finfo(np.float64).eps else 0.0
    return 1.0 - residual_sum / total_sum


def weighted_within_percent(errors, threshold, weights):
    within = np.asarray(errors, dtype=np.float64) <= float(threshold)
    return 100.0 * float(np.average(within.astype(np.float64), weights=weights))


def regression_metrics(contract, actual_residual, predicted_residual, weights, *, scale=None):
    model_error = absolute_residual_error(contract, actual_residual, predicted_residual)
    baseline_error = absolute_residual_error(
        contract, actual_residual, np.zeros_like(actual_residual)
    )
    baseline_mae = float(np.average(baseline_error, weights=weights))
    model_mae = float(np.average(model_error, weights=weights))
    result = {
        "samples": len(model_error),
        "baseline_mae": baseline_mae,
        "model_mae": model_mae,
        "baseline_rmse": float(np.sqrt(np.average(np.square(baseline_error), weights=weights))),
        "model_rmse": float(np.sqrt(np.average(np.square(model_error), weights=weights))),
        "baseline_residual_r2": weighted_r2(actual_residual, np.zeros_like(actual_residual), weights),
        "model_residual_r2": weighted_r2(actual_residual, predicted_residual, weights),
        "unweighted_model_mae": float(np.mean(model_error)),
        "model_median_absolute_error": float(np.median(model_error)),
        "model_p90_absolute_error": float(np.quantile(model_error, 0.90)),
        "model_p95_absolute_error": float(np.quantile(model_error, 0.95)),
        "relative_mae_improvement_percent": (
            100.0 * (baseline_mae - model_mae) / baseline_mae
            if baseline_mae > 0.0 else 0.0
        ),
    }
    if scale is not None:
        pixel_error = model_error * np.asarray(scale)
        baseline_pixel_error = baseline_error * np.asarray(scale)
        result.update({
            "baseline_within_5pct_reference_percent": weighted_within_percent(baseline_error, 0.05, weights),
            "model_within_5pct_reference_percent": weighted_within_percent(model_error, 0.05, weights),
            "baseline_within_10pct_reference_percent": weighted_within_percent(baseline_error, 0.10, weights),
            "model_within_10pct_reference_percent": weighted_within_percent(model_error, 0.10, weights),
            "baseline_mae_px": float(np.average(baseline_pixel_error, weights=weights)),
            "model_mae_px": float(np.average(pixel_error, weights=weights)),
            "baseline_rmse_px": float(np.sqrt(np.average(np.square(baseline_pixel_error), weights=weights))),
            "model_rmse_px": float(np.sqrt(np.average(np.square(pixel_error), weights=weights))),
            "model_p95_absolute_error_px": float(np.quantile(pixel_error, 0.95)),
            "baseline_within_5px_percent": weighted_within_percent(baseline_pixel_error, 5.0, weights),
            "model_within_5px_percent": weighted_within_percent(pixel_error, 5.0, weights),
            "baseline_within_10px_percent": weighted_within_percent(baseline_pixel_error, 10.0, weights),
            "model_within_10px_percent": weighted_within_percent(pixel_error, 10.0, weights),
        })
    else:
        result.update({
            "baseline_within_2deg_percent": weighted_within_percent(baseline_error, 2.0, weights),
            "model_within_2deg_percent": weighted_within_percent(model_error, 2.0, weights),
            "baseline_within_5deg_percent": weighted_within_percent(baseline_error, 5.0, weights),
            "model_within_5deg_percent": weighted_within_percent(model_error, 5.0, weights),
        })
    return result


def metric_rows(frame, predictions: Mapping[str, np.ndarray], *, split, source_dataset="all"):
    rows = []
    for name, contract in OUTPUT_CONTRACTS.items():
        scale = (
            frame[contract.scale_column].to_numpy()
            if contract.scale_column is not None else None
        )
        rows.append({
            "split": split,
            "source_dataset": source_dataset,
            "output": name,
            "unit": contract.unit,
            **regression_metrics(
                contract,
                frame[contract.residual_column].to_numpy(),
                predictions[name],
                frame["sample_weight"].to_numpy(),
                scale=scale,
            ),
        })
    return rows


def prediction_frame(frame, predictions: Mapping[str, np.ndarray], *, split):
    metadata = [
        "sample_id", "group_id", "image_path", "source_dataset", "view",
        "target_annotation_id", "target_chain_rank", "chain_count", "sample_weight",
    ]
    result = frame[metadata].copy()
    result.insert(1, "split", split)
    for name, contract in OUTPUT_CONTRACTS.items():
        actual_residual = frame[contract.residual_column].to_numpy()
        predicted_residual = np.asarray(predictions[name])
        baseline = frame[contract.baseline_column].to_numpy()
        predicted_value = baseline + predicted_residual
        if name == "orientation":
            predicted_value = wrap_axial_deg(predicted_value)
        result[f"{name}_actual"] = frame[contract.actual_column].to_numpy()
        result[f"{name}_baseline"] = baseline
        result[f"{name}_predicted"] = predicted_value
        result[f"{name}_absolute_error"] = absolute_residual_error(
            contract, actual_residual, predicted_residual
        )
    return result

## 2. Load the fixed dataset splits

The loader checks required columns, finite model values, positive weights, unique sample IDs, and group separation between partitions. Only the feature names declared in `feature_schema.json` will enter XGBoost.

In [ ]:
frames, schema, feature_columns = load_dataset(
    DATASET_ROOT, limit_per_split=LIMIT_PER_SPLIT
)

split_summary = pd.DataFrame({
    split: {
        "samples": len(frame),
        "groups": frame["group_id"].nunique(),
        "images": frame["image_path"].nunique(),
    }
    for split, frame in frames.items()
}).T
display(split_summary)
print(f"XGBoost receives {len(feature_columns)} numerical features.")

In [ ]:
feature_map = pd.DataFrame({
    "column": feature_columns,
    "context": [name.split("_")[1] for name in feature_columns],
    "measurement": ["_".join(name.split("_")[2:]) for name in feature_columns],
})
display(feature_map)

metadata_columns = schema.get("metadata_columns", [])
print("Metadata retained for auditing but excluded from XGBoost:")
print(metadata_columns)

## 3. Explicit leakage audit

No target (`y_*`), interpolation baseline, identifier, image, source, or view column may appear in the model matrix.

In [ ]:
forbidden_prefixes = ("y_", "baseline_")
assert not any(name.startswith(forbidden_prefixes) for name in feature_columns)
assert not set(feature_columns) & {
    "sample_id", "group_id", "image_path", "source_dataset_key", "view"
}

group_sets = {split: set(frame["group_id"]) for split, frame in frames.items()}
assert group_sets["train"].isdisjoint(group_sets["val"])
assert group_sets["train"].isdisjoint(group_sets["test"])
assert group_sets["val"].isdisjoint(group_sets["test"])

for split, frame in frames.items():
    assert np.isfinite(frame[feature_columns].to_numpy()).all()

print("Leakage audit passed. Only four-context morphology enters XGBoost.")

## 4. Inspect group weights

Each known patient/case—or radiograph when no patient ID exists—has total weight 1 within its split. This prevents long annotated chains from dominating the fit.

In [ ]:
weight_checks = []
for split, frame in frames.items():
    group_weight = frame.groupby("group_id")["sample_weight"].sum()
    weight_checks.append({
        "split": split,
        "min_group_weight": group_weight.min(),
        "max_group_weight": group_weight.max(),
    })
display(pd.DataFrame(weight_checks))

## 5. Measure the interpolation baseline

A zero residual means accepting the `m1/p1` interpolation unchanged. XGBoost must beat this baseline to justify its added complexity. Dimension errors are normalized; orientation errors are degrees.

In [ ]:
baseline_rows = []
for split in ("val", "test"):
    frame = frames[split]
    for name, contract in OUTPUT_CONTRACTS.items():
        actual_residual = frame[contract.residual_column].to_numpy()
        baseline_error = absolute_residual_error(
            contract, actual_residual, np.zeros_like(actual_residual)
        )
        baseline_rows.append({
            "split": split,
            "output": name,
            "unit": contract.unit,
            "weighted_baseline_mae": np.average(
                baseline_error, weights=frame["sample_weight"]
            ),
        })
baseline_table = pd.DataFrame(baseline_rows)
display(baseline_table.round(5))

## 6. Configure the five regressors

The validation set controls early stopping. Keep the test set out of parameter selection.

In [ ]:
MODEL_PARAMS = {
    "objective": "reg:squarederror",
    "eval_metric": "mae",
    "n_estimators": 1200,
    "learning_rate": 0.03,
    "max_depth": 3,
    "min_child_weight": 4.0,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.0,
    "reg_lambda": 2.0,
    "tree_method": "hist",
    "device": "cpu",
    "early_stopping_rounds": 60,
    "random_state": SEED,
    "n_jobs": 4,
}
display(pd.Series(MODEL_PARAMS, name="value").to_frame())

## 7. Train one model per residual target

Run this cell to fit all five models. On the current dataset this normally takes only seconds on CPU.

In [ ]:
X_train = frames["train"][feature_columns]
X_val = frames["val"][feature_columns]
X_test = frames["test"][feature_columns]
train_weights = frames["train"]["sample_weight"].to_numpy()
val_weights = frames["val"]["sample_weight"].to_numpy()

models = {}
predictions = {"val": {}, "test": {}}
histories = {}
importance_rows = []

for name, contract in OUTPUT_CONTRACTS.items():
    print(f"Training {name}...")
    model = XGBRegressor(**MODEL_PARAMS)
    model.fit(
        X_train,
        frames["train"][contract.residual_column],
        sample_weight=train_weights,
        eval_set=[(X_val, frames["val"][contract.residual_column])],
        sample_weight_eval_set=[val_weights],
        verbose=False,
    )
    models[name] = model
    predictions["val"][name] = model.predict(X_val)
    predictions["test"][name] = model.predict(X_test)
    histories[name] = model.evals_result()
    importance_rows.extend(
        {"output": name, "feature": feature, "importance": float(value)}
        for feature, value in zip(feature_columns, model.feature_importances_)
    )
    print(f"  best iteration: {model.best_iteration}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (name, history) in zip(axes.flat, histories.items()):
    values = history["validation_0"]["mae"]
    ax.plot(values, color="#219ebc")
    ax.axvline(models[name].best_iteration, color="#fb8500", linestyle="--")
    ax.set_title(name)
    ax.set_xlabel("Boosting iteration")
    ax.set_ylabel("Validation MAE")
for ax in axes.flat[len(histories):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Compare XGBoost against interpolation

Positive improvement means XGBoost reduced weighted MAE relative to interpolation.

In [ ]:
all_metric_rows = []
prediction_tables = []
for split in ("val", "test"):
    all_metric_rows.extend(metric_rows(frames[split], predictions[split], split=split))
    prediction_tables.append(
        prediction_frame(frames[split], predictions[split], split=split)
    )

metrics = pd.DataFrame(all_metric_rows)
prediction_report = pd.concat(prediction_tables, ignore_index=True)
display(metrics[[
    "split", "output", "baseline_mae", "model_mae",
    "model_rmse", "model_residual_r2",
    "model_p95_absolute_error", "relative_mae_improvement_percent"
]].round(4))

### Regression accuracy within useful tolerances

For dimensions, accuracy means the weighted percentage within 5% and 10% of the patient-specific reference height or width. For orientation, it means the weighted percentage within 2° and 5°. Pixel metrics remain supplemental because resolution and magnification vary. These thresholds are reporting choices, not clinical acceptance criteria.

In [ ]:
dimension_accuracy = metrics.loc[metrics["unit"].eq("normalized"), [
    "split", "output",
    "baseline_within_5pct_reference_percent", "model_within_5pct_reference_percent",
    "baseline_within_10pct_reference_percent", "model_within_10pct_reference_percent",
    "model_rmse", "model_residual_r2",
]]
orientation_accuracy = metrics.loc[metrics["unit"].eq("degrees"), [
    "split", "output", "baseline_within_2deg_percent", "model_within_2deg_percent",
    "baseline_within_5deg_percent", "model_within_5deg_percent",
    "model_rmse", "model_residual_r2",
]]
display(dimension_accuracy.round(2))
display(orientation_accuracy.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, split in zip(axes, ("val", "test")):
    table = metrics.loc[metrics.split.eq(split)].set_index("output")
    table[["baseline_mae", "model_mae"]].plot.bar(
        ax=ax, color=["#9aa0a6", "#219ebc"]
    )
    ax.set_title(f"{split}: interpolation vs XGBoost")
    ax.set_ylabel("MAE (normalized; orientation is degrees)")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 9. Inspect performance by source

Source is used only for reporting; it was not provided to XGBoost. Small source subsets should be interpreted cautiously.

In [ ]:
source_rows = []
for split in ("val", "test"):
    frame = frames[split]
    for source in sorted(frame["source_dataset"].unique()):
        mask = frame["source_dataset"].eq(source).to_numpy()
        source_rows.extend(metric_rows(
            frame.loc[mask],
            {name: values[mask] for name, values in predictions[split].items()},
            split=split,
            source_dataset=source,
        ))
source_metrics = pd.DataFrame(source_rows)
display(source_metrics[[
    "split", "source_dataset", "output", "samples",
    "baseline_mae", "model_mae", "relative_mae_improvement_percent"
]].round(4))

## 10. Inspect feature importance

Built-in importance is exploratory. Correlated morphology features can share importance, so do not interpret it as clinical causality.

In [ ]:
feature_importance = pd.DataFrame(importance_rows)
mean_importance = (feature_importance.groupby("feature")["importance"]
                   .mean().sort_values(ascending=False).head(15))
ax = mean_importance.sort_values().plot.barh(figsize=(8, 5), color="#219ebc")
ax.set_title("Top mean feature importance across five models")
ax.set_xlabel("Built-in importance")
plt.tight_layout()
plt.show()
display(feature_importance.sort_values(["output", "importance"], ascending=[True, False]).groupby("output").head(5))

## 11. Inspect the largest errors

Use image path and annotation ID to review landmark mistakes, unusual projections, or uncommon morphology.

In [ ]:
ERROR_OUTPUT = "left_height"  # Change to any trained output.
largest_errors = (prediction_report.loc[prediction_report.split.eq("test")]
                  .nlargest(20, f"{ERROR_OUTPUT}_absolute_error"))
display(largest_errors[[
    "sample_id", "source_dataset", "image_path",
    "target_annotation_id", "target_chain_rank", "chain_count",
    f"{ERROR_OUTPUT}_actual", f"{ERROR_OUTPUT}_baseline",
    f"{ERROR_OUTPUT}_predicted", f"{ERROR_OUTPUT}_absolute_error",
]])

## 12. Save the trained models and reports

In [ ]:
if SAVE_ARTIFACTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model_dir = OUTPUT_DIR / "models"
    model_dir.mkdir(exist_ok=True)
    for name, model in models.items():
        model.save_model(model_dir / f"{name}.json")

    metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    source_metrics.to_csv(OUTPUT_DIR / "metrics_by_source.csv", index=False)
    prediction_report.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

    with (OUTPUT_DIR / "training_history.json").open("w", encoding="utf-8") as file:
        json.dump(histories, file, indent=2)
        file.write("\n")
    manifest = {
        "task": "four-context observed masked morphology residual regression",
        "research_use_only": True,
        "dataset_root": str(DATASET_ROOT),
        "feature_columns": feature_columns,
        "model_parameters": MODEL_PARAMS,
        "split_rows": {split: len(frame) for split, frame in frames.items()},
        "best_iterations": {name: int(model.best_iteration) for name, model in models.items()},
    }
    with (OUTPUT_DIR / "run_manifest.json").open("w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)
        file.write("\n")
    print("Saved artifacts to:", OUTPUT_DIR)

## 13. Verify saved-model reload

In [ ]:
if SAVE_ARTIFACTS:
    for name in OUTPUT_CONTRACTS:
        reloaded = XGBRegressor()
        reloaded.load_model(OUTPUT_DIR / "models" / f"{name}.json")
        check = reloaded.predict(X_test.head(10))
        assert np.isfinite(check).all()
    print("All five saved models reload and predict successfully.")

## Next experiments

1. Inspect large errors before changing parameters.
2. Tune only against validation data; avoid repeatedly optimizing on test.
3. Compare annotation-landmark context with CenterNet-predicted context.
4. Add uncertainty or abstention before any downstream screening use.